# Using Pydantic for State Validation in LangGraph

In LangGraph, the **State Schema** defines the structure and types of data that flow between nodes.
LangGraph supports three primary ways to define your State:
1. **`TypedDict`**: Lightweight dictionaries with static type hints (no runtime validation).
2. **`dataclass`**: Standard Python data classes with object-oriented fields and default values.
3. **`pydantic.BaseModel`**: Production-grade data validation, runtime type enforcement, and automatic coercion.

---

### Why use Pydantic for LangGraph State?

| Feature | `TypedDict` | `dataclass` | `Pydantic BaseModel` |
| :--- | :--- | :--- | :--- |
| **Runtime Type Validation** | ❌ None (ignored at runtime) | ❌ None (ignored by default) | ✅ **Strict runtime enforcement** |
| **Data Coercion / Preprocessing** | ❌ Manual | ❌ Manual | ✅ **Built-in (`@field_validator`)** |
| **Field Constraints** (min/max, regex) | ❌ No | ❌ No | ✅ **Supported (`Field(...)`)** |
| **State Access in Nodes** | `state["key"]` (dict indexing) | `state.key` (dot notation) | `state.key` (dot notation) |
| **Overhead** | Minimal | Low | Slight validation overhead |

> **Best Practice Tip:** Use **Pydantic** when your graph receives external inputs (API payloads, user forms), processes LLM structured outputs, or requires defensive validation across mission-critical steps.

---
## 1. Defining a Pydantic State Schema & Normal Invocation

To use Pydantic, define a class that inherits from `pydantic.BaseModel`:
- LangGraph automatically validates inputs against the schema when entering the graph.
- Inside node functions, the state is passed as a Pydantic model instance, allowing clean attribute access (`state.name`).
- Nodes return a dictionary with the updated fields.

In [1]:
from langgraph.graph import StateGraph, START, END
from pydantic import BaseModel

# 1. Define the State Schema using Pydantic BaseModel
class State(BaseModel):
    name: str

# 2. Define node function (receives state as a Pydantic model instance)
def example_node(state: State):
    print("Type of state inside node:", type(state))
    print("State name attribute:", state.name)
    # Access fields using dot notation (state.name)
    return {"name": f"Hello, {state.name}!"}

# 3. Build and compile the StateGraph
builder = StateGraph(State)
builder.add_node("example_node", example_node)
builder.add_edge(START, "example_node")
builder.add_edge("example_node", END)
graph = builder.compile()

# 4. Invoke graph with valid input
result = graph.invoke({"name": "Arpit"})
print("✅ Graph Result:", result)

Type of state inside node: <class '__main__.State'>
State name attribute: Arpit
✅ Graph Result: {'name': 'Hello, Arpit!'}


---
## 2. Runtime Validation in Action: Catching `ValidationError`

What happens if invalid input is passed into the graph?
- Because `name` is declared as a `str`, passing an integer like `{"name": 123}` triggers a `pydantic.ValidationError`.
- LangGraph intercepts this before the node executes, preventing corrupt data from propagating.
- By wrapping `graph.invoke(...)` in a `try...except ValidationError` block, we can gracefully catch and inspect detailed error diagnostics (`e.errors()`).

In [2]:
from langgraph.graph import StateGraph, START, END
from pydantic import BaseModel, ValidationError

class State(BaseModel):
    name: str

def example_node(state: State):
    return {"name": f"Hello, {state.name}!"}

builder = StateGraph(State)
builder.add_node("example_node", example_node)
builder.add_edge(START, "example_node")
builder.add_edge("example_node", END)
graph = builder.compile()

# Passing invalid input: int instead of str
try:
    graph.invoke({"name": 123})  # <-- Deliberately invalid input
except ValidationError as e:
    print("❌ Graph execution failed due to invalid state input:")
    for error in e.errors():
        field_name = error['loc'][0]
        error_msg = error['msg']
        input_val = error['input']
        print(f" - Field '{field_name}': {error_msg} (Received: {input_val})")

❌ Graph execution failed due to invalid state input:
 - Field 'name': Input should be a valid string (Received: 123)


---
## 3. Automatic Data Coercion with `@field_validator`

In real-world applications, you may want the graph to be resilient and automatically coerce or sanitize inputs rather than failing outright.

With Pydantic's `@field_validator(..., mode="before")`:
- `mode="before"` intercepts raw values **before** strict type validation is enforced.
- We can automatically convert numbers or other types to `str` (or perform trimming, casing, etc.).
- Now, `graph.invoke({"name": 123})` succeeds smoothly!

In [3]:
from langgraph.graph import StateGraph, START, END
from pydantic import BaseModel, field_validator

class State(BaseModel):
    name: str

    # Validator that runs BEFORE Pydantic's internal type checking
    @field_validator("name", mode="before")
    @classmethod
    def convert_to_string(cls, v):
        # Automatically converts numbers or other types to str
        return str(v) if v is not None else ""

def example_node(state: State):
    return {"name": f"Hello, {state.name}!"}

builder = StateGraph(State)
builder.add_node("example_node", example_node)
builder.add_edge(START, "example_node")
builder.add_edge("example_node", END)
graph = builder.compile()

# Invoking with integer 123 succeeds because convert_to_string casts it to "123"
result = graph.invoke({"name": 123})
print("✅ Result with Validator Coercion:", result)

✅ Result with Validator Coercion: {'name': 'Hello, 123!'}


---
## 4. Comparison: `TypedDict` (No Runtime Validation)

Why didn't `TypedDict` catch type errors in earlier examples?
- Python's `TypedDict` is purely for **static type checking** (linters, IDE autocomplete, and Mypy).
- At runtime, Python treats `TypedDict` as a regular dictionary and **does not validate types**.
- When you pass `{"name": 123}` into a `TypedDict` graph, it executes without error, but the integer `123` flows unchecked through your graph — which can cause unexpected runtime crashes in later nodes (e.g., string slicing, `.lower()`, or prompt templates).

In [4]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END

# State defined using TypedDict
class State(TypedDict):
    name: str  # Type hint says str, but Python won't enforce this at runtime

def example_node(state: State):
    # TypedDict state uses dictionary key access
    return {"name": f"Hello, {state['name']}!"}

builder = StateGraph(State)
builder.add_node("example_node", example_node)
builder.add_edge(START, "example_node")
builder.add_edge("example_node", END)
graph = builder.compile()

# Passing an integer 123 into a TypedDict graph
# Runs without raising any validation errors at runtime!
response = graph.invoke({"name": 123})
print("⚠️ TypedDict Output (executed without validation):", response)
print(f"Data type of 'name' in state: {type(response['name'])}")

⚠️ TypedDict Output (executed without validation): {'name': 'Hello, 123!'}
Data type of 'name' in state: <class 'int'>


---
## 5. Summary & Key Takeaways

1. **Strict Safety with Pydantic**: When you need strict schema compliance and runtime validation, use `pydantic.BaseModel`. It guarantees that all inputs and intermediate states match the expected types.
2. **Graceful Error Handling**: Catch `pydantic.ValidationError` around `.invoke()` calls to deliver helpful feedback or route to recovery fallback nodes.
3. **Data Sanitization**: Leverage `@field_validator(..., mode="before")` to clean, coerce, and normalize inputs before validation checks run.
4. **TypedDict vs Pydantic**:
   - Use **`TypedDict`** for simple graphs, maximum speed, and minimal boilerplate when inputs are already trusted.
   - Use **`Pydantic`** when dealing with user-facing input, external APIs, complex nested models, or defensive data validation.